[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DhiSys-AI/MRM-Trace/blob/main/notebooks/003%20-%20Real%20Collection%20Walkthrough.ipynb)

# mrm-trace · 003: Real Collection Walkthrough

This notebook shows the **real collection pipeline** using `memray` — no model files or root access needed.

We run a Python workload that simulates LLM memory patterns (weight loading + KV cache generation),
capture it with `MemrayRunner`, parse the raw binary into the mrm-trace canonical format,
and inspect what the trace looks like before analysis.

> **Part A — Real collection:** actual `.bin` file captured and parsed  
> **Part B — What perf mem adds:** comparison with hardware PMU collection  
> **Part C — Real model guide:** instructions for running against an actual LLM  

### Requirements

| Component | Needed for |
|---|---|
| `memray` Python package | Part A (install cell below) |
| Linux (Colab standard runtime ✓) | memray with `native_traces=True` |
| Root / sudo | `native_traces=True` and `perf mem` — Colab standard runtime has root |
| GGUF model file | Part C only (not needed for this notebook) |

## 1. Install

In [ ]:
!pip install -q mrm-trace memray

In [ ]:
import os, sys, platform
import mrm_trace

print(f"mrm-trace   : {mrm_trace.__version__}")
print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.system()} {platform.release()}")

try:
    is_root = (os.getuid() == 0)
except AttributeError:
    is_root = False  # Windows
print(f"Root access : {is_root}")

try:
    import memray
    print(f"memray      : {memray.__version__}")
except ImportError:
    print("memray      : NOT INSTALLED — run the install cell above")

print()
if is_root:
    print("native_traces=True is available (C-level allocations will be captured).")
else:
    print("Running as non-root — using native_traces=False (Python-level allocations only).")
    print("numpy array allocations are C-level and will NOT be captured.")
    print("We will use bytearray for the simulation instead.")

## 2. Simulated LLM Workload

We define two functions that mimic the memory patterns of an LLM:

| Function | Simulates | Pattern |
|---|---|---|
| `load_weights()` | Model weight loading | Large allocation, long-lived |
| `generate_tokens()` | Token generation with KV cache | Many small allocs, quickly freed |

With `native_traces=True` (available on Colab as root), memray captures numpy's C-level
allocations. With `native_traces=False`, we use `bytearray` (Python-level) instead.

The key insight: **weights live for the full inference duration; KV blocks are short-lived**.
This is exactly the `high_mrm` vs `low_mrm` distinction mrm-trace is designed to measure.

In [ ]:
import numpy as np

def load_weights():
    """
    Simulate loading model weights.
    Returns a list of weight matrices (long-lived allocations).
    In a real run: ggml_init allocates gigabytes of weight tensors here.
    """
    # 8 layers × 512x512 float32 = 8 MB total
    layers = [np.random.randn(512, 512).astype(np.float32) for _ in range(8)]
    return layers


def generate_tokens(weights, n_tokens=40):
    """
    Simulate token generation.
    Each token allocates a KV block and frees it (eviction), simulating
    a bounded KV cache. Reads weights on every step.
    In a real run: llama_kv_cache_update drives this pattern.
    """
    for i in range(n_tokens):
        k = np.random.randn(32, 64).astype(np.float16)   # KV key block
        v = np.random.randn(32, 64).astype(np.float16)   # KV value block
        _ = weights[i % len(weights)][:32, :64] @ k.T    # attention
        del k, v                                           # explicit eviction


print("Simulation functions defined.")
print(f"  load_weights()  : 8 × 512x512 float32 matrices = ~8 MB")
print(f"  generate_tokens(): {40} tokens, 2 × (32×64 float16) = ~8 KB per token")

## 3. Capture with MemrayRunner

`MemrayRunner` wraps `memray.Tracker`. It records every allocation and deallocation
that occurs inside its start/stop window, writing a binary `.bin` file.

**What memray records:**
- Address of each allocation
- Allocator type (malloc, new, free, delete, …)
- Call stack at the time of allocation

**What memray does NOT record (perf mem gives these):**
- Individual load/store instructions at hardware level
- Access latency (L1/L2/L3/DRAM)
- NUMA node, CPU core
- Timestamps at nanosecond precision

In [ ]:
import pathlib

output_dir = pathlib.Path("/content/raw_output")
output_dir.mkdir(exist_ok=True)
print("Output directory:", output_dir)

In [ ]:
from mrm_trace.collector.memray_runner import MemrayRunner

try:
    is_root = (os.getuid() == 0)
except AttributeError:
    is_root = False

runner = MemrayRunner(native_traces=is_root)  # True on Colab (root), False elsewhere

if not runner.is_available():
    raise RuntimeError("memray is not installed. Run: pip install memray")

print(f"Starting capture (native_traces={is_root}) ...")
runner.start(target_pids=[], output_dir=output_dir)

# ── Phase 1: weight loading ──────────────────────────────────────────────────
weights = load_weights()

# ── Phase 2: generation ─────────────────────────────────────────────────────
generate_tokens(weights, n_tokens=40)

# ── Phase 3: teardown ───────────────────────────────────────────────────────
del weights

meta = runner.stop()
print(f"Capture complete.")
print(f"  File : {meta.output_file}")
print(f"  Size : {meta.output_file_size_bytes:,} bytes")

## 4. Parse the Raw Trace

`parse_memray()` is a streaming generator that reads the binary `.bin` file and
yields one canonical row dict per allocation record.

It maps allocator types to mrm-trace op types:
- `malloc`, `new`, etc. → `op_type = "store"` (memory is being written)
- `free`, `delete`, etc. → `op_type = "load"` (read-before-free)

In [ ]:
from mrm_trace.parser.memray_parser import parse_memray
import pandas as pd

memray_bin = output_dir / "memray.bin"

rows = list(parse_memray(memray_bin, run_id="demo_memray_001"))
trace_raw = pd.DataFrame(rows)

print(f"Parsed {len(trace_raw):,} allocation records")
print(f"Columns: {list(trace_raw.columns)}")
print()
print(f"op_type distribution:")
print(trace_raw["op_type"].value_counts().to_string())

In [ ]:
print("Sample raw rows (addresses, symbols, region_type):")
display(trace_raw[["sample_id", "address", "address_page", "op_type", "symbol", "dso", "region_type"]].head(15))

## 5. Understanding the Raw Trace

Look at the `symbol` and `dso` columns — they show **Python and numpy internals**,
not the `ggml_init` / `llama_kv_cache_update` symbols a real LLM would produce.

| What you see (simulation) | What a real run shows |
|---|---|
| `ndarray.__new__`, `PyMem_Malloc` | `ggml_init`, `gguf_load_tensor` |
| `<ipython-input-...>` DSO | `libggml.so`, `libllama.so` |
| `region_type = unknown` | `weight`, `kv_cache`, `activation` |
| `timestamp_ns = 0` | Hardware timestamps (nanoseconds) |

The `region_type` is `unknown` because the labeller's symbol rules are
written for real LLM binaries. The labeller classification rule is:

```python
# Simplified example from labeller/symbol_rules.py
if "ggml_init" in symbol or "gguf_load" in symbol:    region_type = "weight"
elif "kv_cache" in symbol:
    region_type = "kv_cache"
```

For our simulation, we can inspect the call stack manually.

In [ ]:
print("Top 15 symbols in the trace:")
print(trace_raw["symbol"].value_counts().head(15).to_string())
print()
print("Top DSOs:")
print(trace_raw["dso"].value_counts().head(10).to_string())
print()
print("region_type distribution:")
print(trace_raw["region_type"].value_counts().to_string())

## 6. Label with TraceLabeller

Even with `unknown` region types, we can run the labeller to produce
a `region_map` and `kv_lifecycle`. Rows with `region_type` already set
are passed through unchanged — the labeller supplements missing fields.

In [ ]:
from mrm_trace.labeller import TraceLabeller

labeller = TraceLabeller()
labelled_rows = list(labeller.label(trace_raw.to_dict(orient="records")))
labelled = pd.DataFrame(labelled_rows)

region_map   = pd.DataFrame(labeller.region_map())
kv_lifecycle = pd.DataFrame(labeller.kv_lifecycle())

print(f"Labelled trace : {labelled.shape}")
print(f"Region map     : {region_map.shape}  rows")
print(f"KV lifecycle   : {kv_lifecycle.shape} rows")
print()
print("Sample region map entries:")
if not region_map.empty:
    cols = [c for c in ["region_id", "region_type", "access_count"] if c in region_map.columns]
    display(region_map[cols].head(10))

## 7. What perf mem Adds

memray captures allocation events (malloc/free). `perf mem` captures every
**hardware load and store instruction** at PMU speed. The difference:

| Capability | memray | perf mem |
|---|---|---|
| Allocation events (malloc/free) | ✅ | ❌ |
| Hardware load / store events | ❌ | ✅ |
| Access latency (cycles/ns) | ❌ | ✅ |
| Cache level (L1/L2/L3/DRAM) | ❌ | ✅ |
| NUMA node | ❌ | ✅ |
| Nanosecond timestamps | ❌ | ✅ |
| Root / CAP_PERFMON required | No | Yes |
| Works in containers / VMs | Yes | WSL2 + host PMU only |
| Overhead | Low | Low (sampling) |
| Trace size (70B model) | ~1 GB | 10–50 GB |

**Which to use?**
- Research / paper figures → `perf mem` (hardware PMU, authoritative data)
- CI / cloud environments → `memray` (no root, works anywhere)
- Always → `process_monitor` runs in parallel as a coarse RSS/CPU baseline

## 8. Running Against a Real Model

This section is a **reference guide** — it requires Linux, a GGUF model file,
and (for perf mem) root access.

### Step 1: Download a small model

```bash
# TinyLlama-1.1B in 4-bit quantisation (~670 MB)
wget -P /content/models \
  https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
```

### Step 2: Write a config

```yaml
experiment:
  name: "tinyllama_mrm_baseline"

models:
  - id: "tinyllama-1b-q4"
    engine: "llamacpp"
    path: "/content/models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"
    family: "llama"
    params_billions: 1.1
    quantisation: "Q4_K_M"

sweep:
  context_lengths: [512, 2048]
  repetitions: 2
  warmup_runs: 1

collector:
  primary: "memray"           # Use perf_mem if you have root + Linux PMU
  fallback: "process_monitor"
  memray:
    native_traces: true         # Requires root — captures C-level allocs
```

### Step 3: Validate and run

```bash
# Validate config
mrm-trace validate --config config.yaml

# Preview run list
mrm-trace plan --config config.yaml

# Run collection (Linux + root for perf_mem; memray works without)
mrm-trace run --config config.yaml
```

### Step 4: Analyse the results

After collection, use the Python API (see notebook 001) to label,
analyse, and export the run directory:

```python
from mrm_trace.parser.memray_parser import parse_memray
from mrm_trace.labeller import TraceLabeller
from mrm_trace.analyser import compute_all
from mrm_trace.reporter import RunExporter

# Parse
rows = list(parse_memray("results/.../raw/memray.bin", run_id="run_001"))
import pandas as pd
trace = pd.DataFrame(rows)

# Label, analyse, export  (same as notebook 001)
labeller = TraceLabeller()
labelled = pd.DataFrame(list(labeller.label(trace.to_dict(orient="records"))))
results  = compute_all(labelled)

exporter = RunExporter("results/tinyllama-1b-q4/run_001")
exporter.export(labelled, pd.DataFrame(labeller.region_map()),
                pd.DataFrame(labeller.kv_lifecycle()), results,
                metadata={"run_id": "run_001"}, run_id="run_001")
```

## Next Steps

| Goal | How |
|---|---|
| Full analysis walkthrough | [001 - Getting Started](001%20-%20Getting%20Started.ipynb) |
| Write & validate a config | [002 - YAML Config & Experiment Planning](002%20-%20YAML%20Config%20%26%20Experiment%20Planning.ipynb) |
| Report a bug | [github.com/DhiSys-AI/MRM-Trace/issues](https://github.com/DhiSys-AI/MRM-Trace/issues) |
| Email | info@dhisys.co.uk |

---
*mrm-trace v0.1.2 · DhiSys AI*